# Хранение данных в реляционных БД. Язык SQL
Данные в приложении нужно хранить так, чтобы они сохранялись между сессиями, были доступны после повторного входа пользователя и не терялись при перезапуске сервиса. Для этого недостаточно временных структур внутри программы: нужен отдельный механизм хранения, который обеспечивает сохранность, корректность и ограничение доступа. Для решения этой задачи используются базы данных, а в качестве основного инструмента работы с реляционными СУБД используется язык SQL.

## Основные понятия

- база данных
- СУБД
- реляционная СУБД
- нереляционные базы данных
- SQL
- декларативный язык
- источник истины
- типы данных
- целостность данных
- ограничения
- внешние ключи
- SQLite
- Postgres
- MongoDB
- Redis
- Docker
- DBeaver
- DataGrip
- PyCharm

## Зачем приложению нужна база данных

Если пользователь приходит в сервис, регистрируется и сохраняет свои настройки, приложение должно вернуть эти данные при следующем входе. Это особенно важно, если пользователь заходит с другого устройства: сегодня с компьютера, завтра с телефона. Сервер может вернуть сохранённые данные только в том случае, если они были записаны на стороне бэкенда и доступны после проверки прав доступа.

Пользовательские данные нужно хранить надёжно и безопасно. Потеря данных означает, что пользователь будет вынужден вводить информацию заново. Раскрытие данных означает уже не просто неудобство, а нарушение безопасности. Поэтому данные должны храниться так, чтобы приложение не теряло их при обновлениях и перезапусках, а доступ к ним получали только те, кому он действительно разрешён.

По этой причине нельзя использовать для долговременного хранения обычный словарь в программе. Если приложение завершит работу, данные, записанные только в память процесса, исчезнут. Словари и другие структуры в коде остаются полезными для промежуточной передачи данных, но не подходят как средство постоянного хранения пользовательской информации.

## Что требуется от хранения данных

Надёжное хранение означает не только сохранность, но и доверие к самим данным. Если поле должно содержать число, дата должна быть датой, а значение скидки — числом в допустимых пределах, база должна уметь это контролировать. Типы данных и ограничения позволяют хранить не произвольный набор значений, а данные, которым приложение действительно может доверять.

Данные нужно разделять по сущностям. Информация о пользователе и информация о его комментариях или действиях не должны без необходимости дублироваться в каждой записи. Если пользователь изменит имя, аватарку или другие атрибуты, правильнее обновить одну запись пользователя, чем переписывать все комментарии, посты и сообщения, которые с ним связаны.

Разделение данных помогает и с безопасностью. Данные о платёжных токенах, паспортных сведениях и других чувствительных сущностях нужно хранить отдельно и ограничивать к ним доступ. При этом база должна сохранять связь между этими данными и конкретной учётной записью, чтобы пользователь получал только свои записи и не видел чужие.

## Реляционные и нереляционные базы данных

Данные можно хранить в реляционных и нереляционных СУБД. Когда речь идёт о реляционной базе данных, фактически имеется в виду реляционная система управления базами данных. Внутри СУБД создаются базы данных, внутри баз данных — таблицы, а внутри них уже хранятся записи.

К нереляционным решениям относятся разные типы хранилищ. Это могут быть документоориентированные базы, key-value-хранилища, графовые, векторные и колоночные решения. Для разных задач используются разные подходы, но основной интерес в данной теме сосредоточен на реляционных СУБД и SQL.

### MongoDB как пример документоориентированного подхода

MongoDB хранит данные в виде документов. Документ представляет собой структуру наподобие JSON-объекта: ключи, значения, вложенные объекты. Такой формат удобен, когда нужно сохранить целостный набор данных, например тело запроса, ответ API или другую документную структуру.

Для хранения пользовательских данных такой подход не всегда подходит как основной. Если база должна оставаться главным источником истины, важны строгие типы, предсказуемая структура и более жёсткий контроль целостности. Поэтому документоориентированное хранение здесь упоминается как отдельный класс решений, а не как основная модель для темы занятия.

### Redis как key-value-хранилище

Redis удобно воспринимать как удалённый словарь по схеме «ключ — значение». Значение можно быстро записать по ключу и затем быстро получить обратно. Это делает Redis удобным для кэшей, временных данных и других значений, которые нужно быстро сохранять и быстро извлекать.

Обычно Redis используют не для основной предметной модели приложения, а как вспомогательное хранилище. Его сильная сторона — скорость чтения и записи. Поэтому Redis подходит для кэширования и временных ключей, а не как основное средство построения реляционной модели пользовательских данных.

## Почему в качестве основных решений выбираются SQLite и Postgres

Из реляционных СУБД в качестве основных решений выбираются SQLite и Postgres. Существуют и другие системы, например MySQL, MariaDB, Oracle и MS SQL, но для практической работы в рамках этой темы достаточно опоры на два основных варианта: SQLite и Postgres.

Если база данных нужна по умолчанию и никаких специальных требований пока нет, разумно брать SQLite. Если позже становится понятно, что приложению нужен сетевой доступ к базе, отдельный сервер, управление доступами и удалённое подключение, тогда нужен Postgres.

## SQLite

SQLite — это библиотека, которая реализует полноценную реляционную базу данных и может использоваться прямо из Python без отдельной установки серверной СУБД. В стандартной поставке Python уже есть модуль для работы со SQLite, поэтому Python можно установить вместе с уже готовым инструментом для работы с реляционной базой данных.

SQLite хранит всю базу в одном файле. Это сразу задаёт и преимущества, и ограничения. С одной стороны, решение оказывается очень простым: база лежит рядом с приложением, не требует отдельного серверного окружения и хорошо подходит для небольших и обычных прикладных задач. С другой стороны, одновременная работа нескольких приложений с одним и тем же файлом создаёт ограничения по записи и может приводить к конфликтам.

Для обычных приложений SQLite часто оказывается достаточно. Простые сайты, Telegram-боты, интернет-магазины и другие прикладные сервисы вполне могут работать с SQLite, если не требуется удалённая серверная БД и сложная схема доступа. Ограничение SQLite связано прежде всего с тем, что база остаётся файлом на диске и должна находиться рядом с приложением, которое с ней работает.

Если окружение не позволяет приложению писать на диск, либо приложение разворачивается так, что ему нужен удалённый доступ к базе данных, SQLite уже не подходит как основное решение. В таком случае нужен отдельный сервер базы данных.

## Postgres

Postgres — это реляционная СУБД, которая используется как серверное решение. К ней можно подключаться по сети, управлять пользователями, разграничивать доступ и строить более гибкую инфраструктуру хранения. Если приложению нужен именно удалённый доступ к базе данных, удобнее сразу использовать Postgres.

Postgres поддерживает современные типы данных и расширенные возможности работы с ними. В частности, в нём есть удобная работа с JSON и отдельный тип `JSONB`, который обеспечивает более эффективную обработку таких данных. Кроме того, Postgres позволяет задавать ограничения, работать с доступами и использовать более гибкую серверную конфигурацию, чем SQLite.

Если SQLite выбирается как решение по умолчанию, то Postgres выбирается там, где требуется сетевой режим работы, отдельный сервер, доступ нескольких компонентов приложения к одной базе и управление правами пользователей.

## Почему реляционная СУБД важна для целостности данных

Реляционная СУБД позволяет гарантировать целостность данных за счёт типов, ограничений и связей. Можно ограничить длину строки, запретить `NULL`, задать уникальность, проверить допустимые значения, ограничить диапазон чисел и использовать внешние ключи для ссылок между таблицами.

Если считать базу данных главным источником истины, то в неё не должно попадать то, что само приложение потом посчитает неправильным. Если поле должно содержать телефонный номер, база не должна принимать заведомо некорректное значение, если это значение потом предполагается использовать как настоящий телефон. Если поле хранит скидку, можно ограничить его допустимым диапазоном. Если одна таблица ссылается на другую, связь можно защитить внешним ключом.

При этом степень строгости зависит от архитектурного решения. Базу можно использовать как простое хранилище с минимальными ограничениями, а можно как слой, который сам поддерживает значительную часть целостности. В рамках этой темы предпочтение отдаётся подходу, где ограничения, типы и связи по возможности задаются на стороне базы данных.

## SQL как язык работы с реляционной базой данных

SQL — это Structured Query Language, язык структурированных запросов. Его назначение состоит в том, чтобы описывать, какие данные нужно получить или как нужно изменить данные в базе. Это декларативный язык: в запросе задаётся желаемый результат, а не пошаговый алгоритм его достижения.

У разных СУБД есть свои диалекты SQL. Стандарт SQL существует, но каждая система реализует его по-своему и дополняет своими возможностями. Поэтому базовые запросы в целом узнаваемы между разными СУБД, однако детали работы с типами, JSON, датой и временем, транзакциями и другими механизмами могут отличаться.

В практической работе это означает, что знание SQL даёт общую основу для понимания запросов в разных реляционных СУБД, но при переходе к особенностям конкретной системы нужно учитывать её диалект.

## Базовые примеры SQL

SQL удобно начинать с простых запросов, которые не требуют таблиц и позволяют увидеть сам принцип работы языка.

```sql
select 1;
select version();
select uuidv7();
select now();
```

Такие запросы показывают, что SQL может возвращать литералы, вызывать функции и выдавать служебную информацию о текущем соединении и сервере.

Алиасы позволяют переименовать выражение в результирующем наборе данных.


In [ ]:
select 1 as one;
select 1 + 2 as "three";
select 1 + 2 as "1 + 2 =";

SyntaxError: invalid syntax (1476580495.py, line 1)

## Создание таблицы

Для хранения данных сначала нужна таблица. Таблица `notes` создаётся с полями `id`, `title`, `note` и `created_at`.

```sql
create table if not exists notes (
    id serial primary key,
    title text not null unique,
    note text not null default '',
    created_at timestamptz not null default now()
);
```
Здесь задаётся сразу несколько правил. `id` создаётся как первичный ключ. `title` не может быть пустым и должен быть уникальным. `note` не может быть пустым и по умолчанию получает пустую строку. `created_at` автоматически получает текущее время.

Если таблицу нужно удалить и создать заново, используется `drop table`.

```sql
drop table notes;
```

## Вставка данных

Одна запись вставляется так:

```sql
insert into notes (title, note)
values ('sql intro', 'my thoughts on the sql intro');
```
Несколько записей можно вставить одним запросом:

```sql
insert into notes (title, note)
values
    ('python intro', 'some python tricks.'),
    ('js intro', 'frontend most used language.');
```
Такой формат позволяет за один запрос добавить несколько строк в таблицу.

## Чтение данных, сортировка и фильтрация

Для получения всех данных из таблицы используется `select *`.

```sql
select * from notes;
```
Для сортировки применяется `order by`. Например, можно отсортировать записи по идентификатору в обратном порядке:
```sql
select * from notes
order by id desc;
```
Или отсортировать по тексту заметки:
```sql
select * from notes
order by note;
```
Фильтрация выполняется через `where`. Для сопоставления со строковым шаблоном используется `like`.
```sql
select * from notes
where note like '%.';
```
Для поиска без учёта регистра можно использовать `ilike`.
```sql
select * from notes
where title ilike '%js%';
```
## Генерация набора данных

Postgres позволяет генерировать данные запросом. Для этого можно использовать `generate_series` и формировать значения на основе последовательности чисел.

```sql
select
    'title ' || s as title,
    'note ' || s as note
from generate_series(1, 100000) as s;
```
Такой запрос создаёт большой набор строк, который затем можно использовать как источник данных, в том числе для последующей вставки.

## Как запускать и проверять запросы

SQLite можно быстро проверять прямо в браузере через онлайн-демонстрации. Postgres удобно запускать в Docker и подключать к нему клиентский инструмент. Для работы с базой подходят DBeaver, DataGrip или PyCharm Professional. Эти инструменты позволяют создавать подключения, выполнять запросы, просматривать таблицы и проверять результат работы SQL-кода.

Если нужен локальный экземпляр Postgres, его удобно запускать в Docker, а затем подключаться к нему через графический клиент. Такой способ позволяет быстро получить рабочую среду для практики без ручной установки серверной СУБД в систему.

## Выводы

- Пользовательские данные нельзя хранить только внутри памяти приложения, если они должны сохраняться между сессиями и устройствами.
- Для корректного хранения важны не только сохранность данных, но и типы, ограничения, целостность и разделение по сущностям.
- Из нереляционных решений отдельно полезны MongoDB и Redis, но основной акцент делается на реляционных СУБД.
- SQLite подходит как решение по умолчанию, если база может находиться рядом с приложением и не нужен сетевой доступ.
- Postgres нужен там, где требуется отдельный сервер базы данных, удалённое подключение и более гибкое управление доступами.
- SQL — это декларативный язык запросов, который позволяет создавать таблицы, вставлять данные, выбирать их, сортировать и фильтровать.